In [ ]:
from gurobipy import read

In [ ]:
m = read('a1q1.lp')

In [ ]:
m.optimize()   # optimize model m

In [ ]:
m.printAttr('X')   # print variable values in the optimal solution of m

In [ ]:
import math
from gurobipy import GRB

m.setParam('Method', 0) # simplex so sensitivity attributes are available
m.optimize()

zA = m.getVarByName("z_A")
lo, hi = zA.SAObjLow, zA.SAObjUp   # allowable range for the coef of z_A

print("Continuous range keeping current basis:", lo, "to", hi)

# Integer p that keep the same optimal solution (inclusive)
p_min = math.ceil(lo)
p_max = math.floor(hi)
print("Integer p range (inclusive):", p_min, "to", p_max)

In [ ]:
from gurobipy import GRB

def plan_sig(model, tol=0):
    return tuple((v.VarName, round(v.X, 6)) for v in model.getVars() if abs(v.X) > tol)

m.setParam('OutputFlag', 0)
m.optimize()

baseline_sig = plan_sig(m)

zA = m.getVarByName("z_A")

P_LO, P_HI = 0, 10000
intervals_by_sig = {}

prev_sig = None
start = None

for p in range(P_LO, P_HI + 1):
    zA.Obj = p
    m.update()
    m.optimize()
    sig = plan_sig(m)

    if prev_sig is None:
        prev_sig, start = sig, p
    elif sig != prev_sig:
        intervals_by_sig.setdefault(prev_sig, []).append((start, p-1))
        prev_sig, start = sig, p

if prev_sig is not None:
    intervals_by_sig.setdefault(prev_sig, []).append((start, P_HI))

base_intervals = intervals_by_sig.get(baseline_sig, [])
print("Baseline plan kept for p in:", base_intervals)

other_plans = [(sig, ivals) for sig, ivals in intervals_by_sig.items() if sig != baseline_sig]
print(f"Other distinct plans encountered: {len(other_plans)}")

for k, (sig, ivals) in enumerate(other_plans[:3], 1):
    print(f"Alt plan #{k} intervals: {ivals}")
